# 🧵 Saree Design Retrieval — Kaggle Notebook

This notebook trains and evaluates an EfficientNet-B0 embedding model for saree surface-design retrieval. It reports gallery identification and pair verification.

> **Before running:** publish the project files to the GitHub repository configured below, create a Kaggle Notebook from this file, attach the DeepLure handloom dataset as a private input, and enable Internet so Kaggle can clone the code and fetch pretrained weights. Do not publish or redistribute the proprietary image dataset.

The design labels are provisional visual groupings. The small results are exploratory, not production accuracy. See `RESULTS.md` in the repository for protocol and limitations.

In [ ]:
from pathlib import Path
import subprocess, sys, json

REPO_URL = 'https://github.com/alo7lika/Saree-design-retrieval.git'
REPO_DIR = Path('/kaggle/working/Saree-design-retrieval')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

# Support either a repo rooted at the project folder or the parent repo layout.
if (REPO_DIR / 'train.py').exists():
    PROJECT = REPO_DIR
elif (REPO_DIR / 'outputs' / 'saree_design_id' / 'train.py').exists():
    PROJECT = REPO_DIR / 'outputs' / 'saree_design_id'
else:
    raise FileNotFoundError('Training code not found in cloned repository. Push the saree_design_id project files to GitHub first.')

print('Project folder:', PROJECT)
print('Files:', ', '.join(sorted(p.name for p in PROJECT.iterdir())[:30]))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)

## 📁 Attach the dataset

In Kaggle, use **Add Input** to attach the handloom images from the provided DeepLure corpus. Keep the dataset private. The cell below finds a known manifest image and sets the image root automatically; it stops with a clear error if the input has not been attached.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
hits = list(INPUT_ROOT.rglob('h_img_34590.jpg'))
if not hits:
    raise FileNotFoundError('Attach the DeepLure handloom dataset in Kaggle Add Input; expected to find h_img_34590.jpg.')
DATA_ROOT = hits[0].parent
MANIFEST = PROJECT / 'manifest_deeplure_visual.csv'
assert MANIFEST.is_file(), f'Missing manifest: {MANIFEST}'
print('Image root:', DATA_ROOT)
print('Manifest:', MANIFEST)
print('Image count:', len(list(DATA_ROOT.glob('*.jpg'))))

## 🏋️ Train

The settings below reproduce the short CPU proof-of-concept run. Kaggle GPU can reduce runtime. The first run downloads torchvision's public ImageNet EfficientNet-B0 weights.

In [ ]:
RUN_DIR = Path('/kaggle/working/saree_run')
train_cmd = [sys.executable, str(PROJECT / 'train.py'),
    '--manifest', str(MANIFEST), '--data-root', str(DATA_ROOT),
    '--output-dir', str(RUN_DIR), '--epochs', '5', '--batch-size', '32',
    '--batches-per-epoch', '10', '--workers', '0', '--finetune-last-blocks', '2']
print(' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT, check=True)

## 📊 Evaluate

Evaluation reports retrieval and verification scores, audits gallery/query design and palette overlap, and measures model-only latency. No image examples are displayed or exported.

In [ ]:
EVAL_DIR = RUN_DIR / 'eval'
eval_cmd = [sys.executable, str(PROJECT / 'evaluate.py'),
    '--manifest', str(MANIFEST), '--data-root', str(DATA_ROOT),
    '--checkpoint', str(RUN_DIR / 'best.pt'), '--output-dir', str(EVAL_DIR)]
print(' '.join(eval_cmd))
subprocess.run(eval_cmd, cwd=PROJECT, check=True)
metrics_file = EVAL_DIR / 'metrics.json'
if metrics_file.exists():
    print(json.dumps(json.loads(metrics_file.read_text()), indent=2))
print('Evaluation files:', [p.name for p in EVAL_DIR.iterdir()])

## ⚠️ Interpretation

The primary benchmark has only 20 visually grouped identities total and 4 held-out test designs; cross-palette verification has 16 positive pairs from 3 designs. Labels have not been independently confirmed by textile experts. Report the exact output from this run and keep these limitations visible. These results are exploratory evidence only.